# Base territorial SITP: recorrido reproducible
Ejecutar primero `python -m src.pipeline` desde la raíz. Seleccionar el entorno del proyecto como kernel (instalar `ipykernel` si VS Code lo solicita). Este notebook explica resultados derivados de datos públicos; no contiene visitas de campo.

**Pregunta:** ¿qué controles se necesitan antes de usar el inventario para preparar inspecciones?

In [ ]:
from pathlib import Path
import json
import geopandas as gpd
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'reports/calidad.json').exists(), 'Ejecutar el pipeline primero'
metrics = json.loads((ROOT / 'reports/calidad.json').read_text(encoding='utf-8'))
pd.Series(metrics).drop('hashes_insumos')

## 1. ¿Por qué validar más que la topología?
Los puntos con valores extremos pasan la comprobación topológica, pero no representan coordenadas utilizables. La bitácora identifica el registro, la evidencia de origen y la reconstrucción aplicada.

In [ ]:
pd.read_csv(ROOT / 'reports/correcciones_geometria.csv')

## 2. ¿Qué quedó sin resolver?
Tres puntos no intersectan los polígonos y un cuarto difiere de sus atributos. Se preservan para revisión. Una coordenada plausible no demuestra precisión en terreno.

In [ ]:
pending = pd.read_csv(ROOT / 'reports/pendientes_revision.csv')
pending[['cenefa', 'localidad', 'estado_territorial', 'diferencia_atributos_m']]

## 3. Conservación y cartografía
Cada paradero se conserva una vez. Los conteos incluyen las 20 localidades, incluso si una tiene cero coincidencias. Los puntos sin localidad permanecen fuera de ese agregado.

In [ ]:
points = gpd.read_file(ROOT / 'data/processed/base_territorial.gpkg', layer='paraderos')
counts = pd.read_csv(ROOT / 'reports/paraderos_por_localidad.csv')
assert points.cenefa.is_unique
assert len(points) == int(counts.paraderos.sum()) + metrics['sin_interseccion']
assert points.crs.to_epsg() == 4686
counts.sort_values('paraderos', ascending=False)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports/figures/base_territorial.png')))

## 4. Del inventario al seguimiento
El esquema en `sql/01_esquema_operacion.sql` separa paraderos, visitas e incidencias. La captura, programación de recorridos y los indicadores de ejecución son la siguiente fase. Actualmente no hay registros de visitas reales o simuladas.

Fuentes, licencia y límites: `data/README.md` y `docs/METODOLOGIA.md`.